In [ ]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.widgets import Slider
from matplotlib.patches import FancyArrowPatch
import matplotlib.patheffects as pe


###formatting
BG = "whitesmoke"
PANEL = "white"
BORDER = "silver"
TEXT = "black"
MUTED= "lightgrey"
OPTIMAL = "darkviolet"   ##optimal path
NAIVE = "cyan"           ##naive straight line
CURRENT = "aquamarine"   ##river current arrows
ACCENT = "gold"          ##angle indicator

plt.rcParams.update({
    "figure.facecolor":  BG,
    "axes.facecolor":    PANEL,
    "axes.edgecolor":    BORDER,
    "axes.labelcolor":   TEXT,
    "axes.titlecolor":   TEXT,
    "xtick.color":       MUTED,
    "ytick.color":       MUTED,
    "text.color":        TEXT,
    "grid.color":        BORDER,
    "grid.linewidth":    0.6,
    "font.family":       "sans-serif",
    "font.size":         5,
})

##values
def optimal_angle(V, s):
    ratio = V / s
    if abs(ratio) >= 1:
        return None
    return -np.arcsin(ratio)


def optimal_path(V, s, X=10.0, n=200):

    theta = optimal_angle(V, s)
    if theta is None:
        return None, None, None
    # x-component of ground velocity
    vx = s * np.cos(theta)          # = sqrt(s^2-V^2)
    T  = X / vx                     # travel time
    xs = np.linspace(0, X, n)
    # y stays zero by construction (drift cancelled)
    ys = np.zeros(n)
    return xs, ys, T


def naive_path(V, s, X=10.0, n=200):

    T_naive = X / s                  # time to cross at full speed in x
    xs = np.linspace(0, X, n)
    ts = xs / s
    ys = V * ts                      # pure drift accumulates linearly
    return xs, ys, T_naive


def energy_cost(V, s, X=10.0, exponent=3):
    theta = optimal_angle(V, s)
    if theta is None:
        return None, None
    vx      = s * np.cos(theta)
    T_opt   = X / vx
    T_naive = X / s
    E_opt   = (s ** exponent) * T_opt
    E_naive = (s ** exponent) * T_naive
    return E_opt, E_naive


##plotting
def draw_current_arrows(ax, V, X=10.0, nx=6, ny=4):
    """Draw a grid of current arrows scaled by V."""
    for coll in ax.collections[:]:
        coll.remove()
    for arrow in [c for c in ax.get_children()
                  if isinstance(c, FancyArrowPatch)]:
        arrow.remove()

    if abs(V) < 1e-4:
        return
    xs = np.linspace(0.5, X - 0.5, nx)
    ys = np.linspace(-2.5, 2.5, ny)
    scale = min(abs(V) * 0.35, 0.6)
    for x in xs:
        for y in ys:
            ax.annotate(
                "", xy=(x, y + scale * np.sign(V)),
                xytext=(x, y),
                arrowprops=dict(
                    arrowstyle="->",
                    color=CURRENT,
                    lw=0.8,
                    alpha=0.4,
                ),
            )


def make_angle_arc(ax, theta, radius=1.2):
    """Draw a small arc showing theta* at the origin."""
    if theta is None:
        return []
    angles = np.linspace(0, theta, 60)
    xs = radius * np.cos(angles)
    ys = radius * np.sin(angles)
    arc, = ax.plot(xs, ys, color=ACCENT, lw=1.2, alpha=0.8)
    label = ax.text(
        radius * 0.55, theta / 2 * 0.7,
        f"theta* = {np.degrees(theta):.1f} deg",
        color=ACCENT, fontsize=5, ha="center",
    )
    return [arc, label]


##main figure

def main():
    X   = 10.0   # river width (horizontal distance to target)
    s   = 2.0    # fish swimming speed (fixed)
    V0  = 1.0    # initial current speed

    fig = plt.figure(figsize=(13, 8), facecolor=BG)
    fig.suptitle(
        "Fish Migration Path Optimisation  -  Uniform Cross-flow",
        fontsize=13, color=TEXT, y=0.97, fontfamily="sans-serif",
        fontweight="bold",
    )

    gs = gridspec.GridSpec(
        2, 2,
        figure=fig,
        left=0.07, right=0.97,
        top=0.90, bottom=0.18,
        hspace=0.38, wspace=0.32,
    )

    ax_path  = fig.add_subplot(gs[:, 0])   # main path plot (spans both rows)
    ax_angle = fig.add_subplot(gs[0, 1])   # theta* vs V/s
    ax_cost  = fig.add_subplot(gs[1, 1])   # energy cost vs V/s

    ##axes
    ax_path.set_xlim(-0.5, X + 0.5)
    ax_path.set_ylim(-3.2, 3.2)
    ax_path.set_aspect("equal")
    ax_path.set_xlabel("x  (horizontal distance)", labelpad=6)
    ax_path.set_ylabel("y  (lateral displacement)", labelpad=6)
    ax_path.set_title("Optimal vs naive path", pad=8, fontsize=10)
    ax_path.grid(True, alpha=0.3)
    ax_path.axhline(0, color=BORDER, lw=1.0, zorder=1)

    ##start / end markers
    ax_path.plot(0,  0, "o", color=TEXT,    ms=7,  zorder=5)
    ax_path.plot(X,  0, "*", color=OPTIMAL, ms=12, zorder=5)
    ax_path.text(-0.3, 0,   "Start", color=MUTED, fontsize=6, va="center", ha="right")
    ax_path.text(X+0.2, 0,  "Target", color=OPTIMAL, fontsize=6, va="center")

    ##lines (to be updated)
    opt_line,  = ax_path.plot([], [], color=OPTIMAL, lw=2.5,
                               label="Optimal path  (theta = const)", zorder=4)
    naive_line,= ax_path.plot([], [], color=NAIVE,   lw=2.0,
                               ls="--", label="Naive path  (theta = 0)", zorder=3)
    arrival_dot,= ax_path.plot([], [], "x", color=NAIVE, ms=10, mew=2, zorder=5)

    #heading arrow (optimal)
    heading_arrow = ax_path.annotate(
        "", xy=(0, 0), xytext=(0, 0),
        arrowprops=dict(arrowstyle="-|>", color=OPTIMAL, lw=1.8),
        zorder=6,
    )

    arc_artists = []
    ax_path.legend(
        loc="upper left", fontsize=5,
        facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT,
    )

    ##info box
    info_text = ax_path.text(
        0.98, 0.04, "", transform=ax_path.transAxes,
        fontsize=5, color=TEXT, va="bottom", ha="right",
        bbox=dict(facecolor=PANEL, edgecolor=BORDER, boxstyle="round,pad=0.4"),
    )

    ##angle figure
    ratios = np.linspace(0, 0.99, 300)
    angles = np.degrees(-np.arcsin(ratios))
    ax_angle.plot(ratios, angles, color=OPTIMAL, lw=2)
    ax_angle.set_xlabel("V / s  (current-to-speed ratio)", labelpad=4)
    ax_angle.set_ylabel("Optimal angle theta*  (degrees)", labelpad=4)
    ax_angle.set_title("Optimal heading angle", pad=8, fontsize=8)
    ax_angle.set_xlim(0, 1)
    ax_angle.set_ylim(-92, 2)
    ax_angle.axvline(1, color=NAIVE, lw=1, ls=":", alpha=0.6)
    ax_angle.text(0.96, -45, "infeasible\nV >= s", color=NAIVE,
                  fontsize=4, ha="right", alpha=0.8)
    ax_angle.grid(True, alpha=0.3)
    angle_dot, = ax_angle.plot([], [], "o", color=ACCENT, ms=7, zorder=5)

    ## cost figure
    ratios_c = np.linspace(0, 0.98, 300)
    E_opts, E_naives = [], []
    for r in ratios_c:
        eo, en = energy_cost(r * s, s, X)
        E_opts.append(eo)
        E_naives.append(en)
    E_opts   = np.array(E_opts)
    E_naives = np.array(E_naives)
    ##normalising to E_naive at V=0 
    E0 = E_naives[0]
    ax_cost.plot(ratios_c, E_opts / E0,   color=OPTIMAL, lw=2,
                 label="Optimal path")
    ax_cost.plot(ratios_c, E_naives / E0, color=NAIVE,   lw=2,
                 ls="--", label="Naive path (misses target)")
    ax_cost.set_xlabel("V / s  (current-to-speed ratio)", labelpad=4)
    ax_cost.set_ylabel("Relative energy cost  (x E0)", labelpad=4)
    ax_cost.set_title("Energy cost  ~  s^3 * T", pad=8, fontsize=8)
    ax_cost.set_xlim(0, 1)
    ax_cost.grid(True, alpha=0.3)
    ax_cost.legend(fontsize=7, facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT)
    cost_dot_opt,  = ax_cost.plot([], [], "o", color=OPTIMAL, ms=7, zorder=5)
    cost_dot_naive,= ax_cost.plot([], [], "o", color=NAIVE,   ms=7, zorder=5)

    ##sliders
    ax_sV = fig.add_axes([0.10, 0.09, 0.55, 0.025], facecolor=PANEL)
    ax_ss = fig.add_axes([0.10, 0.04, 0.55, 0.025], facecolor=PANEL)

    sl_V = Slider(ax_sV, "Current  V", 0.0, 1.95,
                  valinit=V0, color=CURRENT)
    sl_s = Slider(ax_ss, "Fish speed  s", 0.5, 4.0,
                  valinit=s,  color=OPTIMAL)

    for sl in (sl_V, sl_s):
        sl.label.set_color(TEXT)
        sl.valtext.set_color(TEXT)

    ##updating

    def update(_=None):
        V_val = sl_V.val
        s_val = sl_s.val
        ratio = V_val / s_val

        ##clear arc artists
        for a in arc_artists:
            try:
                a.remove()
            except Exception:
                pass
        arc_artists.clear()

        ##redraw current arrows
        draw_current_arrows(ax_path, V_val, X)

        theta = optimal_angle(V_val, s_val)

        if theta is None:
            ##infeasible
            opt_line.set_data([], [])
            heading_arrow.set_visible(False)
            arrival_dot.set_data([X], [X * V_val / s_val])  # show where naive ends
            info_text.set_text(
                f"V/s = {ratio:.2f}  -  INFEASIBLE\n"
                f"Current exceeds fish speed"
            )
            angle_dot.set_data([], [])
            cost_dot_opt.set_data([], [])
            cost_dot_naive.set_data([], [])
        else:
            ##optimal path
            xs_opt, ys_opt, T_opt = optimal_path(V_val, s_val, X)
            opt_line.set_data(xs_opt, ys_opt)

            ##heading arrow from origin
            arrow_len = 1.8
            heading_arrow.set_position((0, 0))
            heading_arrow.xy = (
                arrow_len * np.cos(theta),
                arrow_len * np.sin(theta),
            )
            heading_arrow.set_visible(True)

            ##arc
            new_arcs = make_angle_arc(ax_path, theta)
            arc_artists.extend(new_arcs)

            ##naive path
            xs_n, ys_n, T_n = naive_path(V_val, s_val, X)
            naive_line.set_data(xs_n, ys_n)
            arrival_dot.set_data([X], [ys_n[-1]])

            ##info box
            E_opt, E_naive = energy_cost(V_val, s_val, X)
            info_text.set_text(
                f"V/s = {ratio:.2f}\n"
                f"theta* = {np.degrees(theta):.1f} deg\n"
                f"T_opt = {T_opt:.2f}  -  T_naive = {T_n:.2f}\n"
                f"Energy ratio  E_opt/E_naive = {E_opt/E_naive:.3f}"
            )

            ##angle dot
            if ratio <= 0.99:
                angle_dot.set_data([ratio], [np.degrees(theta)])

            ##cost dots: clamp ratio to curve domain [0, 0.98]
            dot_x = min(ratio, 0.98)
            E_dot, E_naive_dot = energy_cost(dot_x * s_val, s_val, X)
            if E_dot is not None:
                cost_dot_opt.set_data([dot_x],   [E_dot       / E0])
                cost_dot_naive.set_data([dot_x], [E_naive_dot / E0])
            else:
                cost_dot_opt.set_data([], [])
                cost_dot_naive.set_data([], [])

        fig.canvas.draw_idle()

    sl_V.on_changed(update)
    sl_s.on_changed(update)
    update()

    plt.show()

    ##keeping references so not garbage collected
    return sl_V, sl_s

plt.show()


if __name__ == "__main__":
    sliders = main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.widgets import Slider
from matplotlib.patches import FancyArrowPatch
from scipy.integrate import solve_ivp
from scipy.optimize import brentq


SHEAR = "darkviolet"    #shear flow optimal path
UNIFORM = "cyan"     #uniform flow comparison path


plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor":   PANEL,
    "axes.edgecolor":   BORDER,
    "axes.labelcolor":  TEXT,
    "axes.titlecolor":  TEXT,
    "xtick.color":      MUTED,
    "ytick.color":      MUTED,
    "text.color":       TEXT,
    "grid.color":       BORDER,
    "grid.linewidth":   0.6,
    "font.family":      "sans-serif",
    "font.size":        9,
})

##calculations

def ode_system(t, state, s, k, V):
    x, y, theta = state
    x_dot     = s * np.cos(theta)
    y_dot     = s * np.sin(theta) + V + k * y
    theta_dot = -(k / 2) * np.sin(2 * theta)
    return [x_dot, y_dot, theta_dot]


def integrate_path(theta0, s, k, V, X=10.0, n_steps=500):

    def hit_target_x(t, state, s, k, V):
        return state[0] - X
    hit_target_x.terminal  = True
    hit_target_x.direction = 1

    T_max = 3 * X / (s * max(np.cos(theta0), 0.05))

    sol = solve_ivp(
        ode_system,
        [0, T_max],
        [0.0, 0.0, theta0],
        args=(s, k, V),
        events=hit_target_x,
        max_step=T_max / n_steps,
        dense_output=True,
    )

    if sol.status == -1 or len(sol.t_events[0]) == 0:
        return None

    T = sol.t_events[0][0]
    ts = np.linspace(0, T, n_steps)
    states = sol.sol(ts)
    return states[0], states[1], states[2], T


def y_at_arrival(theta0, s, k, V, X=10.0):
    result = integrate_path(theta0, s, k, V, X)
    if result is None:
        return np.nan
    _, ys, _, _ = result
    return ys[-1]


def shoot(s, k, V, X=10.0):
    try:
        ##search bracket: fish must angle against combined drift V + k*y
        theta_min = -np.pi / 2 + 0.01
        theta_max =  np.pi / 2 - 0.01

        fa = y_at_arrival(theta_min, s, k, V, X)
        fb = y_at_arrival(theta_max, s, k, V, X)

        if np.isnan(fa) or np.isnan(fb) or fa * fb > 0:
            return None

        theta0 = brentq(y_at_arrival, theta_min, theta_max,
                        args=(s, k, V, X), xtol=1e-6)

        result = integrate_path(theta0, s, k, V, X)
        if result is None:
            return None
        xs, ys, thetas, T = result
        return theta0, xs, ys, thetas, T

    except Exception:
        return None


def uniform_comparison_path(s, k, X=10.0, n=200):
    """
    For comparison: the path if the fish used the uniform-flow
    optimal angle theta* = -arcsin(k * X/2 / s) -- a rough estimate.
    just show the k=0 straight path for contrast.
    """
    xs = np.linspace(0, X, n)
    ys = np.zeros(n)
    return xs, ys


def draw_shear_arrows(ax, k, X=10.0, nx=6, ny=5):
    for coll in ax.collections[:]:
        coll.remove()
    for arrow in [c for c in ax.get_children()
                  if isinstance(c, FancyArrowPatch)]:
        arrow.remove()

    if abs(k) < 1e-4:
        return

    xs = np.linspace(0.8, X - 0.8, nx)
    ys = np.linspace(-3.0, 3.0, ny)
    for x in xs:
        for y in ys:
            speed = k * y
            if abs(speed) < 1e-4:
                continue
            scale = np.clip(abs(speed) * 0.3, 0.05, 0.7)
            ax.annotate(
                "", xy=(x, y + scale * np.sign(speed)),
                xytext=(x, y),
                arrowprops=dict(
                    arrowstyle="->",
                    color=CURRENT,
                    lw=0.7,
                    alpha=0.45,
                ),
            )


##main figure

def main():
    X  = 10.0
    s  = 2.0
    k0 = 0.3
    V0 = 0.8

    fig = plt.figure(figsize=(13, 7), facecolor=BG)
    fig.suptitle(
        "Fish Migration - Sheared Flow  v = V + k*y  (E-L + shooting method)",
        fontsize=11, color=TEXT, y=0.97, fontweight="bold",
    )

    gs = gridspec.GridSpec(2, 2, figure=fig,
                           left=0.07, right=0.97,
                           top=0.90, bottom=0.22,
                           hspace=0.40, wspace=0.32)

    ax_path  = fig.add_subplot(gs[:, 0])
    ax_theta = fig.add_subplot(gs[0, 1])
    ax_cost  = fig.add_subplot(gs[1, 1])

    ax_path.set_xlim(-0.5, X + 0.5)
    ax_path.set_ylim(-4.0, 4.0)
    ax_path.set_aspect("equal")
    ax_path.set_xlabel("x  (horizontal distance)", labelpad=3, fontsize=8)
    ax_path.set_ylabel("y  (lateral displacement)", labelpad=3, fontsize=8)
    ax_path.set_title("Optimal path: sheared vs uniform flow", pad=5, fontsize=9)
    ax_path.grid(True, alpha=0.3)
    ax_path.axhline(0, color=BORDER, lw=1.0, zorder=1)
    ax_path.plot(0, 0, "o", color=TEXT,  ms=7, zorder=5)
    ax_path.plot(X, 0, "*", color=SHEAR, ms=12, zorder=5)
    ax_path.text(-0.3, 0,   "Start",  color=MUTED, fontsize=7, va="center", ha="right")
    ax_path.text(X+0.2, 0, "Target", color=SHEAR, fontsize=7, va="center")

    shear_line,   = ax_path.plot([], [], color=SHEAR,   lw=2.5,
                                  label="Sheared (E-L shooting)", zorder=4)
    uniform_line, = ax_path.plot([], [], color=UNIFORM, lw=1.8,
                                  ls="--", label="Uniform (theta=const)", zorder=3)
    ax_path.legend(loc="upper left", fontsize=7,
                   facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT,
                   borderpad=0.4, labelspacing=0.3)
    info_text = ax_path.text(
        0.98, 0.04, "", transform=ax_path.transAxes,
        fontsize=7, color=TEXT, va="bottom", ha="right", linespacing=1.4,
        bbox=dict(facecolor=PANEL, edgecolor=BORDER, boxstyle="round,pad=0.3"))

    ax_theta.set_xlabel("x  (position along river)", labelpad=2, fontsize=8)
    ax_theta.set_ylabel("theta  (degrees)", labelpad=2, fontsize=8)
    ax_theta.set_title("Heading angle along path", pad=4, fontsize=9)
    ax_theta.set_xlim(0, X)
    ax_theta.set_ylim(-95, 5)
    ax_theta.axhline(0, color=BORDER, lw=0.8, ls=":")
    ax_theta.grid(True, alpha=0.3)
    theta_shear,   = ax_theta.plot([], [], color=SHEAR,   lw=2,
                                    label="Sheared (varies)")
    theta_uniform, = ax_theta.plot([], [], color=UNIFORM, lw=1.5,
                                    ls="--", label="Uniform (const)")
    ax_theta.legend(fontsize=7, facecolor=PANEL, edgecolor=BORDER,
                    labelcolor=TEXT, borderpad=0.4, labelspacing=0.3)

    ax_cost.set_xlabel("Shear rate k", labelpad=2, fontsize=8)
    ax_cost.set_ylabel("Travel time T", labelpad=2, fontsize=8)
    ax_cost.set_title("Travel time vs shear rate", pad=4, fontsize=9)
    ax_cost.set_xlim(0, 0.92)
    ax_cost.grid(True, alpha=0.3)
    cost_line,  = ax_cost.plot([], [], color=SHEAR,   lw=2, label="Sheared")
    cost_uline, = ax_cost.plot([], [], color=UNIFORM, lw=1.5, ls="--", label="Uniform")
    cost_dot,   = ax_cost.plot([], [], "o", color=ACCENT, ms=7, zorder=5)
    ax_cost.legend(fontsize=7, facecolor=PANEL, edgecolor=BORDER,
                   labelcolor=TEXT, borderpad=0.4, labelspacing=0.3)

    ax_sk = fig.add_axes([0.10, 0.13, 0.55, 0.022], facecolor=PANEL)
    ax_sV = fig.add_axes([0.10, 0.08, 0.55, 0.022], facecolor=PANEL)
    ax_ss = fig.add_axes([0.10, 0.03, 0.55, 0.022], facecolor=PANEL)
    sl_k = Slider(ax_sk, "Shear  k",    0.0, 0.90, valinit=k0, color=CURRENT)
    sl_V = Slider(ax_sV, "Current V",   0.0, 1.80, valinit=V0, color=UNIFORM)
    sl_s = Slider(ax_ss, "Fish speed s",0.5, 4.0,  valinit=s,  color=SHEAR)
    for sl in (sl_k, sl_V, sl_s):
        sl.label.set_color(TEXT)
        sl.valtext.set_color(TEXT)

    def update(_=None):
        k_val = sl_k.val
        V_val = sl_V.val
        s_val = sl_s.val

        draw_shear_arrows(ax_path, k_val, X)

        ##uniform comparison
        if V_val < s_val:
            theta_u = -np.arcsin(V_val / s_val)
            uniform_line.set_data(np.linspace(0, X, 200), np.zeros(200))
            T_uniform = X / np.sqrt(s_val**2 - V_val**2)
            theta_uniform.set_data([0, X],
                                   [np.degrees(theta_u), np.degrees(theta_u)])
        else:
            uniform_line.set_data([], [])
            theta_uniform.set_data([], [])
            T_uniform = np.nan

        ##shear flow: one shoot call only
        result = shoot(s_val, k_val, V_val, X)

        if result is None:
            shear_line.set_data([], [])
            theta_shear.set_data([], [])
            cost_dot.set_data([], [])
            info_text.set_text(
                f"k={k_val:.2f}  V={V_val:.2f}  s={s_val:.2f}\n"
                f"No solution found")
        else:
            theta0, xs, ys, thetas, T = result
            shear_line.set_data(xs, ys)
            theta_shear.set_data(xs, np.degrees(thetas))
            cost_dot.set_data([min(k_val, 0.89)], [T])
            info_text.set_text(
                f"k={k_val:.2f}  V={V_val:.2f}  s={s_val:.2f}\n"
                f"theta(0) = {np.degrees(theta0):.1f} deg\n"
                f"theta(T) = {np.degrees(thetas[-1]):.1f} deg\n"
                f"T_shear={T:.3f}  T_uniform={T_uniform:.3f}"
            )

        fig.canvas.draw_idle()

    def update_cost_curve(_=None):
        k_val = sl_k.val
        V_val = sl_V.val
        s_val = sl_s.val
        T_uniform = (X / np.sqrt(s_val**2 - V_val**2)
                     if V_val < s_val else np.nan)

        ks_c = np.linspace(0.01, 0.90, 30)
        Ts_c = [shoot(s_val, ki, V_val, X) for ki in ks_c]
        Ts_c = [r[4] if r is not None else np.nan for r in Ts_c]
        cost_line.set_data(ks_c, Ts_c)
        cost_uline.set_data([0, 0.90], [T_uniform, T_uniform])

        valid = [t for t in Ts_c if not np.isnan(t)]
        if valid:
            ymax = max(max(valid),
                       T_uniform if not np.isnan(T_uniform) else 0) * 1.1
            ax_cost.set_ylim(min(valid) * 0.9, ymax)

        fig.canvas.draw_idle()

    # Fast update on every drag; cost curve only on release
    sl_k.on_changed(update)
    sl_V.on_changed(update)
    sl_s.on_changed(update)

    ##connect to button_release_event on the figure 
    def on_release(event):
        update_cost_curve()
    fig.canvas.mpl_connect("button_release_event", on_release)

    update()
    update_cost_curve()

    return sl_k, sl_V, sl_s


if __name__ == "__main__":
    sliders = main()
    plt.show()